### 1. Load the dataset

In [1]:
from pathlib import Path
import json

import pandas as pd


DATA_DIR = Path("../data/raw")

NODES_PATH = DATA_DIR / "isebel-mecklenburg-nodes.csv"
EDGES_PATH = DATA_DIR / "isebel-mecklenburg-edges.csv"

nodes = pd.read_csv(NODES_PATH)
edges = pd.read_csv(EDGES_PATH)

print("Nodes shape:", nodes.shape)
print("Edges shape:", edges.shape)

print("\nNode columns:")
print(nodes.columns.tolist())

print("\nEdge columns:")
print(edges.columns.tolist())

display(nodes.head())
display(edges.head())

Nodes shape: (20005, 3)
Edges shape: (40604, 4)

Node columns:
['id', 'label', 'properties']

Edge columns:
['src-id', 'dst-id', 'label', 'properties']


,id,label,properties
0,12495,date,"{""datetime"":""1938-10-24"",""__unique"":[""datetime..."
1,10923,date,"{""datetime"":""1925-09-06"",""__unique"":[""datetime..."
2,14299,date,"{""datetime"":""1927-07-12"",""__unique"":[""datetime..."
3,4957,date,"{""datetime"":""1923-09-03"",""__unique"":[""datetime..."
4,2835,date,"{""datetime"":""1935-08-20"",""__unique"":[""datetime..."


,src-id,dst-id,label,properties
0,16006,284,date-of-narration,"{""__label"":""date-of-narration""}"
1,18144,628,date-of-narration,"{""__label"":""date-of-narration""}"
2,3514,1719,date-of-narration,"{""__label"":""date-of-narration""}"
3,3801,3536,date-of-narration,"{""__label"":""date-of-narration""}"
4,49,56,date-of-narration,"{""__label"":""date-of-narration""}"


### 2. Node and relationship type distributions

In [2]:
node_type_counts = (
    nodes["label"]
    .value_counts(dropna=False)
    .rename_axis("node_type")
    .reset_index(name="count")
)

node_type_counts["percentage"] = (
    node_type_counts["count"] / len(nodes) * 100
).round(2)


edge_type_counts = (
    edges["label"]
    .value_counts(dropna=False)
    .rename_axis("edge_type")
    .reset_index(name="count")
)

edge_type_counts["percentage"] = (
    edge_type_counts["count"] / len(edges) * 100
).round(2)

print("Node types:")
display(node_type_counts)

print("Relationship types:")
display(edge_type_counts)

Node types:


,node_type,count,percentage
0,story,14379,71.88
1,keyword,2439,12.19
2,place,1393,6.96
3,person,1088,5.44
4,date,706,3.53


Relationship types:


,edge_type,count,percentage
0,content,30410,74.89
1,place-of-narration,5550,13.67
2,narrator,2365,5.82
3,date-of-narration,2279,5.61


### 3. Parse the JSON properties

In [3]:
def parse_properties(value):
    """Convert a JSON property string into a Python dictionary."""
    if pd.isna(value):
        return {}

    if isinstance(value, dict):
        return value

    try:
        return json.loads(value)
    except (json.JSONDecodeError, TypeError):
        return {"_unparsed_value": value}


nodes = nodes.copy()
edges = edges.copy()

nodes["properties_dict"] = nodes["properties"].apply(parse_properties)
edges["properties_dict"] = edges["properties"].apply(parse_properties)

In [4]:
failed_node_parsing = nodes["properties_dict"].apply(
    lambda value: "_unparsed_value" in value
).sum()

failed_edge_parsing = edges["properties_dict"].apply(
    lambda value: "_unparsed_value" in value
).sum()

print("Unparsed node-property rows:", failed_node_parsing)
print("Unparsed edge-property rows:", failed_edge_parsing)

Unparsed node-property rows: 0
Unparsed edge-property rows: 0


### 4. Inspect node properties

In [5]:
def collect_property_keys(series):
    """Return all property names used in a series of dictionaries."""
    property_keys = set()

    for properties in series:
        if isinstance(properties, dict):
            property_keys.update(properties.keys())

    return sorted(property_keys)


node_property_keys = (
    nodes
    .groupby("label")["properties_dict"]
    .apply(collect_property_keys)
    .reset_index(name="property_keys")
)

display(node_property_keys)

,label,property_keys
0,date,"[__label, __unique, datetime]"
1,keyword,"[__label, __unique, name]"
2,person,"[__label, __unique, gender, name]"
3,place,"[__label, __unique, lat, long, name]"
4,story,"[__label, __unique, path, title]"


In [6]:
for node_type in sorted(nodes["label"].dropna().unique()):
    print(f"\n=== {node_type} ===")

    display(
        nodes.loc[
            nodes["label"] == node_type,
            ["id", "properties_dict"]
        ].head(3)
    )


=== date ===


,id,properties_dict
0,12495,"{'datetime': '1938-10-24', '__unique': ['datet..."
1,10923,"{'datetime': '1925-09-06', '__unique': ['datet..."
2,14299,"{'datetime': '1927-07-12', '__unique': ['datet..."



=== keyword ===


,id,properties_dict
3187,4867,"{'name': 'Tagelöhner', '__unique': ['name'], '..."
3188,928,"{'name': 'buttern', '__unique': ['name'], '__l..."
3189,3914,"{'name': 'Fear', '__unique': ['name'], '__labe..."



=== person ===


,id,properties_dict
706,1589,"{'name': 'Schumacher', 'gender': 'male', '__un..."
707,6581,"{'name': 'Doss', 'gender': 'male', '__unique':..."
708,16737,"{'name': 'Jörn', 'gender': 'female', '__unique..."



=== place ===


,id,properties_dict
1794,1167,"{'name': 'England', 'long': '', 'lat': '', '__..."
1795,2741,"{'name': 'Pritzier', 'long': 'OOPS', 'lat': 'O..."
1796,19779,"{'name': 'Hohen Lukow', 'long': '11.96195', 'l..."



=== story ===


,id,properties_dict
5626,16368,"{'path': 'de.wossidia.xmd-s001-000-007-966', '..."
5627,14282,"{'path': 'de.wossidia.xmd-s001-000-005-437', '..."
5628,8041,"{'path': 'de.wossidia.xmd-s001-000-006-499', '..."


### 5. Data-integrity checks

In [7]:
duplicate_node_ids = nodes["id"].duplicated().sum()

duplicate_node_rows = nodes.duplicated(
    subset=["id", "label", "properties"]
).sum()

duplicate_relationships = edges.duplicated(
    subset=["src-id", "dst-id", "label"]
).sum()

duplicate_edge_rows = edges.duplicated(
    subset=["src-id", "dst-id", "label", "properties"]
).sum()


integrity_summary = pd.Series({
    "missing_node_ids": nodes["id"].isna().sum(),
    "missing_node_labels": nodes["label"].isna().sum(),
    "missing_node_properties": nodes["properties"].isna().sum(),
    "missing_source_ids": edges["src-id"].isna().sum(),
    "missing_destination_ids": edges["dst-id"].isna().sum(),
    "missing_edge_labels": edges["label"].isna().sum(),
    "duplicate_node_ids": duplicate_node_ids,
    "duplicate_node_rows": duplicate_node_rows,
    "duplicate_relationships": duplicate_relationships,
    "duplicate_edge_rows": duplicate_edge_rows,
}, name="count").to_frame()

display(integrity_summary)

print("Node IDs are unique:", nodes["id"].is_unique)

,count
missing_node_ids,0
missing_node_labels,0
missing_node_properties,0
missing_source_ids,0
missing_destination_ids,0
missing_edge_labels,0
duplicate_node_ids,0
duplicate_node_rows,0
duplicate_relationships,0
duplicate_edge_rows,0


Node IDs are unique: True


### 6. Identify the graph schema

In [8]:
id_to_node_type = nodes.set_index("id")["label"]

edges["src_type"] = edges["src-id"].map(id_to_node_type)
edges["dst_type"] = edges["dst-id"].map(id_to_node_type)

relationship_schema = (
    edges
    .groupby(
        ["src_type", "label", "dst_type"],
        dropna=False
    )
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

display(relationship_schema)

,src_type,label,dst_type,count
0,story,content,keyword,30410
3,story,place-of-narration,place,5550
2,story,narrator,person,2365
1,story,date-of-narration,date,2279


In [9]:
unknown_source_nodes = edges["src_type"].isna().sum()
unknown_destination_nodes = edges["dst_type"].isna().sum()

unknown_endpoint_edges = (
    edges["src_type"].isna()
    | edges["dst_type"].isna()
).sum()

print("Edges with unknown source nodes:", unknown_source_nodes)
print("Edges with unknown destination nodes:", unknown_destination_nodes)
print("Edges with an unknown endpoint:", unknown_endpoint_edges)

Edges with unknown source nodes: 0
Edges with unknown destination nodes: 0
Edges with an unknown endpoint: 0


### 7. Create clean node-type tables

In [10]:
def extract_property(dataframe, property_name):
    """Extract one field from the properties dictionary."""
    return dataframe["properties_dict"].apply(
        lambda properties: properties.get(property_name)
    )


stories = nodes[nodes["label"] == "story"].copy()
stories["path"] = extract_property(stories, "path")
stories["title"] = extract_property(stories, "title")


keywords = nodes[nodes["label"] == "keyword"].copy()
keywords["name"] = extract_property(keywords, "name")


people = nodes[nodes["label"] == "person"].copy()
people["name"] = extract_property(people, "name")
people["gender"] = extract_property(people, "gender")


places = nodes[nodes["label"] == "place"].copy()
places["name"] = extract_property(places, "name")
places["longitude_raw"] = extract_property(places, "long")
places["latitude_raw"] = extract_property(places, "lat")

places["longitude"] = pd.to_numeric(
    places["longitude_raw"],
    errors="coerce"
)

places["latitude"] = pd.to_numeric(
    places["latitude_raw"],
    errors="coerce"
)


dates = nodes[nodes["label"] == "date"].copy()
dates["datetime_raw"] = extract_property(dates, "datetime")

dates["datetime"] = pd.to_datetime(
    dates["datetime_raw"],
    errors="coerce"
)

dates["year"] = dates["datetime"].dt.year

### 8. Story metadata coverage

In [11]:
expected_relationships = [
    "content",
    "date-of-narration",
    "narrator",
    "place-of-narration",
]

story_coverage = (
    edges
    .pivot_table(
        index="src-id",
        columns="label",
        values="dst-id",
        aggfunc="nunique",
        fill_value=0
    )
    .reindex(
        index=stories["id"],
        columns=expected_relationships,
        fill_value=0
    )
)

story_coverage.index.name = "story_id"

display(story_coverage.head())

label,content,date-of-narration,narrator,place-of-narration
story_id,,,,
16368,0,0,0,0
14282,0,0,0,0
8041,5,1,0,0
11481,0,0,0,0
12699,7,1,0,5


In [12]:
coverage_summary = pd.DataFrame({
    "stories_with_relation": (story_coverage > 0).sum(),
    "percentage_of_stories": (
        (story_coverage > 0).mean() * 100
    ).round(2),
    "average_values_per_story": story_coverage.mean().round(2),
    "maximum_values_per_story": story_coverage.max()
})

display(coverage_summary)

stories_without_keywords = story_coverage.index[
    story_coverage["content"] == 0
]

print("Total stories:", len(stories))
print("Stories with keywords:", (story_coverage["content"] > 0).sum())
print("Stories without keywords:", len(stories_without_keywords))

,stories_with_relation,percentage_of_stories,average_values_per_story,maximum_values_per_story
label,,,,
content,4577,31.83,2.11,40
date-of-narration,2270,15.79,0.16,2
narrator,2328,16.19,0.16,3
place-of-narration,2652,18.44,0.39,8


Total stories: 14379
Stories with keywords: 4577
Stories without keywords: 9802


### 9. Keyword-frequency analysis

In [13]:
content_edges = (
    edges.loc[
        edges["label"] == "content",
        ["src-id", "dst-id"]
    ]
    .drop_duplicates()
)

keyword_story_counts = (
    content_edges
    .groupby("dst-id")["src-id"]
    .nunique()
    .sort_values(ascending=False)
)

keyword_names = keywords.set_index("id")["name"]

keyword_frequency_table = (
    keyword_story_counts
    .rename("story_count")
    .to_frame()
    .join(keyword_names.rename("keyword"))
    .reset_index(names="keyword_id")
)

In [14]:
display(keyword_story_counts.describe().to_frame())

print(
    "Keywords appearing in one story:",
    (keyword_story_counts == 1).sum()
)

print(
    "Keywords appearing in at least 100 stories:",
    (keyword_story_counts >= 100).sum()
)

print("\nMost frequent keywords:")

display(
    keyword_frequency_table[
        ["keyword_id", "keyword", "story_count"]
    ].head(30)
)

,src-id
count,2439.000000
mean,12.468225
std,48.884527
min,1.000000
25%,1.000000
50%,2.000000
75%,7.000000
max,714.000000


Keywords appearing in one story: 998
Keywords appearing in at least 100 stories: 41

Most frequent keywords:


,keyword_id,keyword,story_count
0,16,Frevelsage,714
1,18,Frevel,714
2,17,sacrilege,714
3,32,Hexe,628
4,35,witches,626
5,29,Hexen,626
6,92,Tote,588
7,91,dead,588
8,93,Toter,557
9,6,historical legends,516


### 10. Find candidate duplicate or translated keywords

In [15]:
keyword_to_stories = (
    content_edges
    .groupby("dst-id")["src-id"]
    .apply(frozenset)
)

groups_by_story_set = {}

for keyword_id, story_set in keyword_to_stories.items():
    groups_by_story_set.setdefault(
        story_set,
        []
    ).append(keyword_id)


candidate_groups = [
    (story_set, keyword_ids)
    for story_set, keyword_ids in groups_by_story_set.items()
    if len(keyword_ids) > 1 and len(story_set) >= 2
]

candidate_group_rows = []

for group_number, (story_set, keyword_ids) in enumerate(
    candidate_groups,
    start=1
):
    for keyword_id in keyword_ids:
        candidate_group_rows.append({
            "group": group_number,
            "keyword_id": keyword_id,
            "keyword": keyword_names.get(keyword_id),
            "story_count": len(story_set)
        })

candidate_keyword_groups = pd.DataFrame(candidate_group_rows)

print(
    "Candidate duplicate or translation groups:",
    len(candidate_groups)
)

display(
    candidate_keyword_groups
    .sort_values(
        ["story_count", "group"],
        ascending=[False, True]
    )
    .head(100)
)

Candidate duplicate or translation groups: 288


,group,keyword_id,keyword,story_count
4,3,16,Frevelsage,714
5,3,17,sacrilege,714
6,3,18,Frevel,714
7,4,29,Hexen,626
8,4,35,witches,626
...,...,...,...,...
242,101,1775,nicht grugen,33
243,101,1776,not to fear,33
32,16,140,Wann,32
33,16,146,When,32


### 11. Geographic data quality

In [16]:
places["valid_coordinates"] = (
    places["longitude"].notna()
    & places["latitude"].notna()
    & places["longitude"].between(-180, 180)
    & places["latitude"].between(-90, 90)
)

coordinate_summary = pd.Series({
    "total_places": len(places),
    "places_with_valid_coordinates": places["valid_coordinates"].sum(),
    "places_without_valid_coordinates": (
        ~places["valid_coordinates"]
    ).sum(),
}, name="count").to_frame()

display(coordinate_summary)

,count
total_places,1393
places_with_valid_coordinates,636
places_without_valid_coordinates,757


In [17]:
valid_place_ids = set(
    places.loc[
        places["valid_coordinates"],
        "id"
    ]
)

place_edges = edges.loc[
    edges["label"] == "place-of-narration",
    ["src-id", "dst-id"]
].copy()

stories_with_any_place = set(place_edges["src-id"])

stories_with_valid_coordinates = set(
    place_edges.loc[
        place_edges["dst-id"].isin(valid_place_ids),
        "src-id"
    ]
)

stories_with_keywords = set(
    story_coverage.index[
        story_coverage["content"] > 0
    ]
)

geographic_analysis_stories = (
    stories_with_keywords
    & stories_with_valid_coordinates
)


geographic_coverage = pd.Series({
    "total_stories": len(stories),
    "stories_with_any_place": len(stories_with_any_place),
    "stories_with_valid_coordinates": (
        len(stories_with_valid_coordinates)
    ),
    "stories_with_keywords": len(stories_with_keywords),
    "stories_with_keywords_and_coordinates": (
        len(geographic_analysis_stories)
    ),
}, name="count").to_frame()

geographic_coverage["percentage_of_all_stories"] = (
    geographic_coverage["count"]
    / len(stories)
    * 100
).round(2)

display(geographic_coverage)

print(
    "Percentage of keyword-annotated stories with coordinates:",
    round(
        len(geographic_analysis_stories)
        / len(stories_with_keywords)
        * 100,
        2
    )
)

,count,percentage_of_all_stories
total_stories,14379,100.00
stories_with_any_place,2652,18.44
stories_with_valid_coordinates,2390,16.62
stories_with_keywords,4577,31.83
stories_with_keywords_and_coordinates,2384,16.58


Percentage of keyword-annotated stories with coordinates: 52.09


### 12. Stories with multiple dates, narrators, or places

In [18]:
relationships_to_check = [
    "date-of-narration",
    "narrator",
    "place-of-narration",
]

multiple_value_summary = []

for relationship in relationships_to_check:
    multiple_value_summary.append({
        "relationship": relationship,
        "stories_with_multiple_values": (
            story_coverage[relationship] > 1
        ).sum(),
        "maximum_values_for_one_story": (
            story_coverage[relationship].max()
        )
    })

multiple_value_summary = pd.DataFrame(
    multiple_value_summary
)

display(multiple_value_summary)

,relationship,stories_with_multiple_values,maximum_values_for_one_story
0,date-of-narration,9,2
1,narrator,35,3
2,place-of-narration,2070,8


In [19]:
for relationship in relationships_to_check:
    print(f"\n=== {relationship} ===")

    display(
        story_coverage.loc[
            story_coverage[relationship] > 1,
            [relationship]
        ]
        .sort_values(
            relationship,
            ascending=False
        )
        .head(10)
    )


=== date-of-narration ===


label,date-of-narration
story_id,
6976,2
6910,2
1022,2
575,2
19227,2
17431,2
1945,2
8130,2
15894,2



=== narrator ===


label,narrator
story_id,
12564,3
13465,3
12584,2
12236,2
6224,2
5429,2
11334,2
660,2
16240,2



=== place-of-narration ===


label,place-of-narration
story_id,
7189,8
15821,7
575,7
16559,7
19988,6
16832,6
3339,6
4143,6
4509,6


In [20]:
def inspect_story_relationships(story_id, relationship):
    """Display all target nodes linked to a story by one relationship."""
    linked_nodes = (
        edges.loc[
            (edges["src-id"] == story_id)
            & (edges["label"] == relationship),
            ["src-id", "label", "dst-id"]
        ]
        .merge(
            nodes[
                ["id", "label", "properties_dict"]
            ],
            left_on="dst-id",
            right_on="id",
            how="left",
            suffixes=("_edge", "_node")
        )
    )

    display(linked_nodes)


# Example:
# inspect_story_relationships(7189, "place-of-narration")

### 13. Date quality

In [21]:
print("Missing or invalid dates:", dates["datetime"].isna().sum())
print("Earliest date:", dates["datetime"].min())
print("Latest date:", dates["datetime"].max())
print("Unique dates:", dates["datetime"].nunique())

Missing or invalid dates: 0
Earliest date: 1016-08-12 00:00:00
Latest date: 2014-04-08 00:00:00
Unique dates: 706


In [22]:
current_year = pd.Timestamp.today().year

dates_requiring_review = dates[
    (dates["year"] < 1800)
    | (dates["year"] > current_year)
].copy()

display(
    dates_requiring_review[
        ["id", "datetime_raw", "datetime", "year"]
    ].sort_values("datetime")
)

,id,datetime_raw,datetime,year
420,12427,1016-08-12,1016-08-12,1016
500,7809,1643-01-01,1643-01-01,1643
160,15209,1683-01-01,1683-01-01,1683


In [23]:
date_relationships = (
    edges.loc[
        edges["label"] == "date-of-narration",
        ["src-id", "dst-id"]
    ]
    .rename(columns={
        "src-id": "story_id",
        "dst-id": "date_id"
    })
)

date_lookup = (
    dates[
        ["id", "datetime_raw", "datetime", "year"]
    ]
    .rename(columns={"id": "date_id"})
)

story_lookup = (
    stories[
        ["id", "title", "path"]
    ]
    .rename(columns={"id": "story_id"})
)

review_date_stories = (
    date_relationships
    .merge(
        date_lookup,
        on="date_id",
        how="left"
    )
    .merge(
        story_lookup,
        on="story_id",
        how="left"
    )
)

review_date_stories = review_date_stories[
    (review_date_stories["year"] < 1800)
    | (review_date_stories["year"] > current_year)
]

display(
    review_date_stories[
        [
            "story_id",
            "title",
            "path",
            "datetime_raw"
        ]
    ].sort_values("datetime_raw")
)

,story_id,title,path,datetime_raw
1050,12426,[10819] Hexen - Blocksberg Orte,de.wossidia.xmd-s001-000-010-819,1016-08-12
76,10189,[12520] Hexen - Blocksberg Varia,de.wossidia.xmd-s001-000-012-520,1643-01-01
2129,7807,[12518] Hexen - Blocksberg Varia,de.wossidia.xmd-s001-000-012-518,1643-01-01
1030,15205,[10798] Hexen - Buhle,de.wossidia.xmd-s001-000-010-798,1683-01-01


### 14. Story title and path quality

In [24]:
stories["title_clean"] = (
    stories["title"]
    .fillna("")
    .astype(str)
    .str.strip()
)

stories["path_clean"] = (
    stories["path"]
    .fillna("")
    .astype(str)
    .str.strip()
)

story_identifier_quality = pd.Series({
    "stories_without_titles": (
        stories["title_clean"] == ""
    ).sum(),
    "stories_without_paths": (
        stories["path_clean"] == ""
    ).sum(),
    "duplicated_nonempty_paths": (
        stories.loc[
            stories["path_clean"] != "",
            "path_clean"
        ].duplicated().sum()
    ),
    "duplicated_nonempty_titles": (
        stories.loc[
            stories["title_clean"] != "",
            "title_clean"
        ].duplicated().sum()
    ),
}, name="count").to_frame()

display(story_identifier_quality)

,count
stories_without_titles,8
stories_without_paths,0
duplicated_nonempty_paths,0
duplicated_nonempty_titles,0


### 15. Final inspection summary

In [25]:
inspection_summary = pd.Series({
    "total_nodes": len(nodes),
    "total_edges": len(edges),
    "total_stories": len(stories),
    "total_keywords": len(keywords),
    "total_people": len(people),
    "total_places": len(places),
    "total_dates": len(dates),
    "keyword_annotated_stories": len(stories_with_keywords),
    "stories_with_valid_coordinates": (
        len(stories_with_valid_coordinates)
    ),
    "stories_usable_for_geographic_analysis": (
        len(geographic_analysis_stories)
    ),
    "duplicate_node_ids": duplicate_node_ids,
    "duplicate_relationships": duplicate_relationships,
    "edges_with_unknown_endpoints": unknown_endpoint_edges,
}, name="value").to_frame()

display(inspection_summary)

,value
total_nodes,20005
total_edges,40604
total_stories,14379
total_keywords,2439
total_people,1088
total_places,1393
total_dates,706
keyword_annotated_stories,4577
stories_with_valid_coordinates,2390
stories_usable_for_geographic_analysis,2384


## Inspection conclusions

- The graph structure is internally consistent: node IDs are unique,
  relationships have valid endpoints, and no duplicate relationships were found.
- The dataset contains 14,379 stories, but only 4,577 stories have keyword
  annotations. The community-detection analysis will therefore focus on this
  annotated subset.
- Keyword frequencies are highly uneven. Many keywords occur in only one story,
  while a small number occur in hundreds of stories.
- Several multilingual keyword labels have identical story assignments. These
  are candidates for consolidation or down-weighting, but they should not be
  merged automatically.
- Geographic analysis is possible for 2,384 stories that have both keyword
  annotations and valid narration-place coordinates.
- Many stories are associated with multiple narration places. The meaning of
  these multiple locations must be examined before calculating geographic
  distances.
- A few unusually early dates require verification, but they do not block the
  keyword-based community-detection analysis.